# Energy profile
Relative free energies from the DFT data and the profile plot. Plot options: `settings.yaml`, pathways: `c8_energy_profiles_settings.yaml`.

In [1]:
import os
from collections import OrderedDict
import numpy as np
import pandas as pd
import yaml
from kinetic_model.energy_profile.energy_processing import calculate_relative_energies
from kinetic_model.energy_profile.energy_profile import EnergyProfile

In [2]:
T, P = 300.0, 101325.0          # K, Pa
order = ["1", "2", "3", "4", "5", "5a", "6", "1p", "7", "8", "9", "5p", "6p", "1pp", "10", "11", "12", "5pp", "6pp", "1ppp",
         "2-3", "3-4", "6-1p", "7-8", "8-9", "6p-1pp", "10-11", "11-12", "6pp-1ppp"]
res = calculate_relative_energies(
    structures_dir=os.path.abspath('../data/structures'), molecules_dir=os.path.abspath('../data/gas-phase-molecules'),
    temperatures=[T], models=['QM-QM_TPSSh-PBED3'], ref_struc='1', pathways=['trigonal'], pressures=[P])
dft = res[res['order'].isin(order)].reset_index(drop=True)
dft[['order', 'system_name', 'rel_E_eV', 'rel_H_eV', 'rel_G_eV']].head()

,order,system_name,rel_E_eV,rel_H_eV,rel_G_eV
0,10,et-Ni-hexyl,-3.448928,-3.105692,-1.52287
1,11,Ni-octyl,-3.830101,-3.472387,-1.850689
2,12,H-Ni-oct,-3.579445,-3.251423,-1.671682
3,1ppp,Ni-ethyl,-3.419552,-3.062189,-1.657535
4,1pp,Ni-ethyl,-2.327866,-2.09007,-1.165745


In [3]:
settings = yaml.safe_load(open('settings.yaml'))
plot_cfg = yaml.safe_load(open('c8_energy_profiles_settings.yaml'))
EP = EnergyProfile(dft, energy_type='Gibbs free energy', energy_unit='kJ/mol')
seqs = [s.split('-->') for s in plot_cfg['sequences']]
colors = plot_cfg['colors']
pos = OrderedDict()
for s in seqs:
    for i, sys_ in enumerate(s):
        pos.setdefault(sys_, 2 * i)
EP.plot()
vals = [v for s in seqs for v in EP.get_energy_range(s)]
pad = 0.1 * (max(vals) - min(vals))
for i, s in enumerate(seqs):
    for sys_ in s:
        EP.add_level(system=sys_, position=pos[sys_], color=colors[i], **({'label': plot_cfg['labels'][i]} if sys_ == s[0] else {}))
        EP.add_intermediate_energy_label(system=sys_, position=pos[sys_], fontsize=18)
        EP.add_structure_name(system=sys_, position=pos[sys_], fontsize=18)
    for a, b in zip(s[:-1], s[1:]):
        EP.add_connection(system1=a, system2=b, color=colors[i])
        EP.add_ts_energy_label(system1=a, system2=b, fontsize=18)
EP.show(xmin=0, xmax=max(pos.values()) + 1, ymin=min(vals), ymax=max(vals), padding=pad, legend=True)